<a href="https://colab.research.google.com/github/Odeyiany2/ACCESS-6.0-Skills-Acquisition-Program-Data-Science/blob/main/Week2_Session4_Data_Cleaning_And_Feature_Creation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Title: Pandas II - Data Cleaning & Feature Creation
**Quick Recap of Last Session:**

Last time, we learned how to:

  * Read CSV files into pandas DataFrames

  * Inspect data using head(), tail(), info(), describe()

  * Select columns and filter rows using loc[] and iloc[]

  * Group data and perform aggregations




**What We Noticed:**
The BudgetWise dataset was messy!

We saw:

* Date formats all over the place (2023-04-25, 08/05/2022, 31-12-23)

* Amount column with currency symbols (₹, $, Rs., INR)

* Missing values in multiple columns

* Category names with typos (Educaton, Fod, Helth)

* Inconsistent location names (BANGALORE, Bangalore, BAN)

**Today's Mission:**

Transform this messy data into clean, analysis-ready data that we can actually use for insights.


 80% of time is spent cleaning data

* Real-world data is collected from different sources
* Human data entry introduces errors
* Systems change over time, creating inconsistencies
* No one designed the data with analysis in mind

What happens if you skip cleaning?

* Wrong calculations (imagine summing "5000" and "₹5000" as strings!)
* Misleading insights
* Failed analyses
* Lost time debugging issues later

In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np


In [ ]:
#Read the dataset
from google.colab import drive
drive.mount('/content/drive')

df = pd.read_csv("/content/drive/My Drive/Colab Notebooks/WEEK2SESSION4/budgetwise_finance_dataset.csv")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
#Get an idea of the dataset
df.head()

,transaction_id,user_id,date,transaction_type,category,amount,payment_mode,location,notes
0,T4999,U018,2023-04-25,Expense,Educaton,3888,card,Ahmedabad,Movie tickets
1,T12828,U133,08/05/2022,Expense,rent,649,NaN,Hyderabad,asdfgh
2,T7403,U091,31-12-23,Income,Freelance,13239,Csh,BAN,Books
3,T12350,U097,NaN,Expense,Fod,6299,Bank Transfer,AHMEDABAD,Electricity bill
4,T7495,U088,10/28/2022,Expense,entertainment,2287,CARD,Hyderabad,NaN


**Missing Values**

Missing values appear as NaN (Not a Number) or None in pandas. They represent data that should exist but doesn't.

In [ ]:
df.isna().sum()

,0
transaction_id,0
user_id,0
date,486
transaction_type,0
category,285
amount,291
payment_mode,808
location,1262
notes,2821


#Strategies for Handling Missing Data

##Strategy 1: Delete Rows or Columns

When to use:

When missing data is less than 5-10% of total

When the column isn't critical for analysis

When you have plenty of other data

**Drop rows with any missing values**

df_clean = df.dropna()

**Drop rows where specific column is missing**

df_clean = df.dropna(subset=['amount'])

**Drop columns with too many missing values**

df_clean = df.drop(columns=['notes'])  # 2821 missing out of 15900


##Strategy 2: Fill Missing Values
When to use:

When you can make reasonable assumptions

When missing data follows a pattern

When you need to preserve all rows

**Fill with a specific value**

df['payment_mode'].fillna('Unknown', inplace=True)

**Fill with the most common value (mode)**

most_common_location = df['location'].mode()[0]

df['location'].fillna(most_common_location, inplace=True)

**Fill numerical columns with mean or median**

df['amount'].fillna(df['amount'].median(), inplace=True)

You Should Use the Median When:
The data contains significant outliers (skewed distribution).

##Strategy 3: Keep and Flag
When to use:

When missing data itself is informative

When you want to analyze patterns in missingness

**Create a flag column**

df['has_notes'] = df['notes'].notna()

Now you can analyze: do transactions with notes differ from those without?

# Let's Clean Our Budgetwise Dataset.

In [ ]:
df.isna().sum()

,0
transaction_id,0
user_id,0
date,486
transaction_type,0
category,285
amount,291
payment_mode,808
location,1262
notes,2821


In [ ]:
print("Missing values per column:")
print(df.isnull().sum())

Missing values per column:
transaction_id         0
user_id                0
date                 486
transaction_type       0
category             285
amount               291
payment_mode         808
location            1262
notes               2821
dtype: int64


In [ ]:
df.head()

,transaction_id,user_id,date,transaction_type,category,amount,payment_mode,location,notes
0,T4999,U018,2023-04-25,Expense,Educaton,3888,card,Ahmedabad,Movie tickets
1,T12828,U133,08/05/2022,Expense,rent,649,NaN,Hyderabad,asdfgh
2,T7403,U091,31-12-23,Income,Freelance,13239,Csh,BAN,Books
3,T12350,U097,NaN,Expense,Fod,6299,Bank Transfer,AHMEDABAD,Electricity bill
4,T7495,U088,10/28/2022,Expense,entertainment,2287,CARD,Hyderabad,NaN


In [ ]:

# NOTES - Too many missing (2821/15900 = 18%), not critical for analysis
# Decision: Drop this column
df_clean = df.drop(columns=['notes'])

# PAYMENT_MODE - 808 missing (5%)
# Decision: Fill with 'Unknown' as it's categorical
df_clean['payment_mode'].fillna('Unknown', inplace=True)

# LOCATION - 1262 missing (8%)
# Decision: Fill with 'Not Specified'
df_clean['location'].fillna('Not Specified', inplace=True)

# DATE - 486 missing (3%)
# Decision: We NEED dates for time-based analysis. Drop these rows.
df_clean = df_clean.dropna(subset=['date'])

# CATEGORY - 285 missing (1.8%)
# Decision: Can't categorize without this. Drop these rows.
df_clean = df_clean.dropna(subset=['category'])

# AMOUNT - 291 missing (1.8%)
# Decision: Can't analyze spending without amounts. Drop these rows.
df_clean = df_clean.dropna(subset=['amount'])

# Step 3: Verify
print(f"\nOriginal shape: {df.shape}")
print(f"Cleaned shape: {df_clean.shape}")
print(f"\nRemaining missing values:")
print(df_clean.isnull().sum())
'''

**Expected Output:**
Original shape: (15900, 9)
Cleaned shape: (14638, 8)  # Lost some rows but gained data quality

'''


Original shape: (15900, 9)
Cleaned shape: (14852, 8)

Remaining missing values:
transaction_id      0
user_id             0
date                0
transaction_type    0
category            0
amount              0
payment_mode        0
location            0
dtype: int64


/tmp/ipython-input-3284950079.py:7: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_clean['payment_mode'].fillna('Unknown', inplace=True)
/tmp/ipython-input-3284950079.py:11: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)

'\n\n**Expected Output:**\nOriginal shape: (15900, 9)\nCleaned shape: (14638, 8)  # Lost some rows but gained data quality\n\n'

In [ ]:
df_clean.head()

,transaction_id,user_id,date,transaction_type,category,amount,payment_mode,location
0,T4999,U018,2023-04-25,Expense,Educaton,3888,card,Ahmedabad
1,T12828,U133,08/05/2022,Expense,rent,649,Unknown,Hyderabad
2,T7403,U091,31-12-23,Income,Freelance,13239,Csh,BAN
4,T7495,U088,10/28/2022,Expense,entertainment,2287,CARD,Hyderabad
5,T12465,U042,04/11/2024,Expense,Foods,4168,Unknown,Not Specified


#Check for Duplicates

In [ ]:
df_clean.duplicated()

,0
0,False
1,False
2,False
4,False
5,False
...,...
15895,False
15896,False
15897,False
15898,True


In [ ]:
df_clean.duplicated().sum()

np.int64(841)

#Removing Duplicate Rows

Rows that appear more than once in your dataset.



In [ ]:
# Remove duplicate rows (keep first occurrence)
df_clean = df_clean.drop_duplicates()


print(f"Shape after removing duplicates: {df_clean.shape}")


Shape after removing duplicates: (14011, 8)



##**Parsing Inconsistent Dates**

**The Problem:**

Looking at our date column, we see chaos:

2023-04-25      # YYYY-MM-DD

08/05/2022      # MM/DD/YYYY

31-12-23        # DD-MM-YY

10/28/2022      # MM/DD/YYYY

NaN             # Missing

In [ ]:
# First, let's see what we're working with
print(df_clean['date'].dtype)  # Will show 'object' (string)

# Attempt to convert with pandas' smart parser
# errors='coerce' turns unparseable dates into NaT (Not a Time)
df_clean['date'] = pd.to_datetime(df_clean['date'], errors='coerce', dayfirst=False)

# Verify the conversion
print(df_clean['date'].dtype)

# Check how many dates failed to parse
unparseable_dates = df_clean['date'].isna().sum()
print(f"Dates that couldn't be parsed: {unparseable_dates}")

if unparseable_dates > 0:
    # Use a more reasonable imputation strategy
    # Fill with the most common date instead of median for better temporal distribution
    mode_date = df_clean['date'].mode()[0] if not df_clean['date'].mode().empty else pd.Timestamp('2023-01-01')
    df_clean['date'] = df_clean['date'].fillna(mode_date)
    #print(f" Filled missing dates with mode date: {mode_date.date()}")

print(f" Date standardization complete")
print(f"Date range: {df_clean['date'].min().date()} to {df_clean['date'].max().date()}")

# Drop rows where date conversion failed
#df_clean = df_clean.dropna(subset=['date'])

# Verify the conversion
print(df_clean['date'].dtype)  # Should show 'datetime64[ns]'
print(df_clean['date'].head())

object
datetime64[ns]
Dates that couldn't be parsed: 8488
 Date standardization complete
Date range: 2021-01-01 to 2024-12-31
datetime64[ns]
0   2023-04-25
1   2021-12-16
2   2021-12-16
4   2021-12-16
5   2021-12-16
Name: date, dtype: datetime64[ns]


In [ ]:
missing_dates = df_clean['date'].isna().sum()
print(f"Total number of missing dates (NaT): {missing_dates}")

Total number of missing dates (NaT): 0


In [ ]:
df_clean.head(10)

,transaction_id,user_id,date,transaction_type,category,amount,payment_mode,location
0,T4999,U018,2023-04-25,Expense,Educaton,3888,card,Ahmedabad
1,T12828,U133,2021-12-16,Expense,rent,649,Unknown,Hyderabad
2,T7403,U091,2021-12-16,Income,Freelance,13239,Csh,BAN
4,T7495,U088,2021-12-16,Expense,entertainment,2287,CARD,Hyderabad
5,T12465,U042,2021-12-16,Expense,Foods,4168,Unknown,Not Specified
6,T4518,U026,2021-12-16,Expense,education,Rs.828,Crd,Lucknow
7,T9824,U061,2024-12-12,Income,Salary,62061,CRD,AHM
8,T0741,U053,2021-12-16,Expense,Utilties,₹5070,upi,KOL
9,T12403,U004,2021-12-16,Income,Others,59543,UPI,LUC
10,T5282,U016,2022-04-15,Expense,Utilties,999999999,Crd,DEL


#**Cleaning the Amount Column**

**Removing Currency Symbols and Converting to Numbers**

**The Problem:**

Our amount column looks like this:
```
3888
649
₹5000
$250
Rs.1500
5459 INR

In [ ]:
df_clean.head()

,transaction_id,user_id,date,transaction_type,category,amount,payment_mode,location
0,T4999,U018,2023-04-25,Expense,Educaton,3888,card,Ahmedabad
1,T12828,U133,2021-12-16,Expense,rent,649,Unknown,Hyderabad
2,T7403,U091,2021-12-16,Income,Freelance,13239,Csh,BAN
4,T7495,U088,2021-12-16,Expense,entertainment,2287,CARD,Hyderabad
5,T12465,U042,2021-12-16,Expense,Foods,4168,Unknown,Not Specified


In [ ]:
df_clean['amount'].value_counts()

,count
amount,
999999,73
-500,62
0,58
999999999,56
-1000,56
...,...
$4262,1
₹5070,1
62061,1


In [ ]:
# First, let's see the data type
print(df_clean['amount'].dtype)

object


In [ ]:

# Step 1: Convert to string (in case some are already numbers)
df_clean['amount'] = df_clean['amount'].astype(str)
#df_clean['amount'] = df_clean['amount'].astype(str)

# Step 2: Remove common currency symbols and text
df_clean['amount'] = df_clean['amount'].str.replace('₹', '', regex=False)
df_clean['amount'] = df_clean['amount'].str.replace('$', '', regex=False)
df_clean['amount'] = df_clean['amount'].str.replace('Rs.', '', regex=False)
df_clean['amount'] = df_clean['amount'].str.replace(' INR', '', regex=False)
df_clean['amount'] = df_clean['amount'].str.replace(',', '', regex=False)  # Remove commas from 1,000
df_clean['amount'] = df_clean['amount'].str.replace('-', '', regex=False)

# regex = False (It only replaces the exact character you asked for).

# Step 3: Remove any remaining whitespace
df_clean['amount'] = df_clean['amount'].str.strip()

# Step 4: Convert to numeric
df_clean['amount'] = pd.to_numeric(df_clean['amount'], errors='coerce')



In [ ]:
# Step 5: Check for values that couldn't convert (will be NaN)
print(f"Values that couldn't convert: {df_clean['amount'].isna().sum()}")


Values that couldn't convert: 0


In [ ]:
print(df_clean['amount'].dtype)

int64


In [ ]:
#Check if there are still negative values
df_clean['amount'].min()


0

In [ ]:
# Step 6: Remove rows where amount is still missing or invalid
df_clean = df_clean.dropna(subset=['amount'])

# Step 7: Remove obviously wrong values (like 999999 )
# Filter out suspiciously high amounts (outliers)
df_clean = df_clean[df_clean['amount'] < 100000]  # Adjust threshold as needed

# Verify
print(df_clean['amount'].dtype)  # Should show float64 or int64
print(df_clean['amount'].describe())

int64
count    13755.000000
mean     12102.550055
std      17244.765771
min          0.000000
25%       2797.500000
50%       5784.000000
75%       9372.500000
max      79999.000000
Name: amount, dtype: float64


In [ ]:
df_clean.head()

,transaction_id,user_id,date,transaction_type,category,amount,payment_mode,location
0,T4999,U018,2023-04-25,Expense,Educaton,3888,card,Ahmedabad
1,T12828,U133,2021-12-16,Expense,rent,649,Unknown,Hyderabad
2,T7403,U091,2021-12-16,Income,Freelance,13239,Csh,BAN
4,T7495,U088,2021-12-16,Expense,entertainment,2287,CARD,Hyderabad
5,T12465,U042,2021-12-16,Expense,Foods,4168,Unknown,Not Specified


In [ ]:
#Checking the category column
df_clean['category'].value_counts().sort_index()

,count
category,
Bonus,373
EDU,210
Education,227
Educaton,216
Entertain,311
Entertainment,278
Entrtnmnt,295
FOOD,512
Fod,470


## **Standardizing Categorical Data**

**Fixing Typos and Inconsistencies**

**The Problem:**

Our category column has variations of the same thing:
```
Food, FOOD, Fod, Foodd, Foods, food
Rent, RENT, Rnt, Rentt, rent
Education, Educaton, EDU, education

In [ ]:
# Step 1: Convert everything to lowercase first
df_clean['category'] = df_clean['category'].str.lower()

# Step 2: Strip whitespace
df_clean['category'] = df_clean['category'].str.strip()

# Step 3: Create a mapping dictionary for common typos/variations
category_mapping = {
    'fod': 'food',
    'foodd': 'food',
    'foods': 'food',
    'rnt': 'rent',
    'rentt': 'rent',
    'educaton': 'education',
    'edu': 'education',
    'helth': 'health',
    'others': 'others',
    'other': 'others',
    'utlities': 'utilities',
    'utilties': 'utilities',
    'utlities': 'utilities',
    'entrtnmnt': 'entertainment',
    'entertain': 'entertainment',
    'traval': 'travel',
    'travl': 'travel',
    'saving': 'savings'
}

# Step 4: Apply the mapping
df_clean['category'] = df_clean['category'].replace(category_mapping)

# Step 5: Check unique categories now
print("Unique categories after cleaning:")
print(df_clean['category'].value_counts())

Unique categories after cleaning:
category
food             2986
rent             2388
travel           1779
entertainment    1176
utilities        1111
education         903
others            599
health            552
investment        418
freelance         405
salary            374
bonus             373
savings           349
utility           300
misc               42
Name: count, dtype: int64


In [ ]:
#Check the location column
df_clean['location'].value_counts().sort_index()

,count
location,
AHM,321
AHMEDABAD,322
Ahmedabad,313
BAN,312
BANGALORE,338
Bangalore,322
CHE,319
CHENNAI,292
Chennai,325


In [ ]:
# Standardize location
df_clean['location'] = df_clean['location'].str.lower().str.strip()

location_mapping = {
    'ban': 'bangalore',
    'mum': 'mumbai',
    'pun': 'pune',
    'del': 'delhi',
    'che': 'chennai',
    'hyd': 'hyderabad',
    'jai': 'jaipur',
    'kol': 'kolkata',
    'luc': 'lucknow',
    'ahm': 'ahmedabad'
}

df_clean['location'] = df_clean['location'].replace(location_mapping)

In [ ]:
df_clean['location'].value_counts()

,count
location,
pune,1325
kolkata,1280
bangalore,1273
chennai,1272
delhi,1269
lucknow,1265
ahmedabad,1262
mumbai,1257
hyderabad,1240


In [ ]:
df_clean.head()

,transaction_id,user_id,date,transaction_type,category,amount,payment_mode,location
0,T4999,U018,2023-04-25,Expense,education,3888,card,ahmedabad
1,T12828,U133,2021-12-16,Expense,rent,649,Unknown,hyderabad
2,T7403,U091,2021-12-16,Income,freelance,13239,Csh,bangalore
4,T7495,U088,2021-12-16,Expense,entertainment,2287,CARD,hyderabad
5,T12465,U042,2021-12-16,Expense,food,4168,Unknown,not specified


#Feature Creation

##Creating New Columns from Existing Data

**What is feature creation?**
Feature creation (or feature engineering) means creating new, useful columns from your existing data. These new features often make analysis easier and reveal insights.

Why create new features?

Raw data rarely comes in the exact form you need for analysis. By creating features, you can:
```
Simplify complex analyses
Reveal patterns
Make visualizations clearer
Enable new types of questions

Common Feature Creation Techniques:
1. Extracting Date Components

Now that date is datetime, we can extract parts
df_clean['year'] = df_clean['date'].dt.year
df_clean['month'] = df_clean['date'].dt.month
df_clean['month_name'] = df_clean['date'].dt.month_name()
df_clean['day_of_week'] = df_clean['date'].dt.day_name()
df_clean['quarter'] = df_clean['date'].dt.quarter

# Check the results
print(df_clean[['date', 'year', 'month', 'month_name', 'day_of_week']].head())

Why this matters:
Now you can answer questions like:

Which month has the highest spending?
Do people spend more on weekends?
What's the quarterly spending trend?

In [ ]:
# Now that date is datetime, we can extract parts
df_clean['year'] = df_clean['date'].dt.year
df_clean['month'] = df_clean['date'].dt.month
df_clean['month_name'] = df_clean['date'].dt.month_name()
df_clean['day_of_week'] = df_clean['date'].dt.day_name()
df_clean['quarter'] = df_clean['date'].dt.quarter

# Check the results
print(df_clean[['date', 'year', 'month', 'month_name', 'day_of_week']].head())

        date  year  month month_name day_of_week
0 2023-04-25  2023      4      April     Tuesday
1 2021-12-16  2021     12   December    Thursday
2 2021-12-16  2021     12   December    Thursday
4 2021-12-16  2021     12   December    Thursday
5 2021-12-16  2021     12   December    Thursday


#Feature Creation - Part 2

**Creating Calculated and Categorical Features**

Creating Spending Categories

In [ ]:
# Categorize amounts into spending levels
def categorize_amount(amount):
    if amount < 1000:
        return 'Small'
    elif amount < 5000:
        return 'Medium'
    else:
        return 'Large'

df_clean['spending_level'] = df_clean['amount'].apply(categorize_amount)

print(df_clean['spending_level'].value_counts())

spending_level
Large     7649
Medium    4897
Small     1209
Name: count, dtype: int64


#Creating Boolean Flags

In [ ]:
# Flag high-value transactions
df_clean['is_high_value'] = df_clean['amount'] > 10000

# Flag weekend transactions
df_clean['is_weekend'] = df_clean['day_of_week'].isin(['Saturday', 'Sunday'])


In [ ]:
df_clean.head()

,transaction_id,user_id,date,transaction_type,category,amount,payment_mode,location,year,month,month_name,day_of_week,quarter,spending_level,is_high_value,is_weekend
0,T4999,U018,2023-04-25,Expense,education,3888,card,ahmedabad,2023,4,April,Tuesday,2,Medium,False,False
1,T12828,U133,2021-12-16,Expense,rent,649,Unknown,hyderabad,2021,12,December,Thursday,4,Small,False,False
2,T7403,U091,2021-12-16,Income,freelance,13239,Csh,bangalore,2021,12,December,Thursday,4,Large,True,False
4,T7495,U088,2021-12-16,Expense,entertainment,2287,CARD,hyderabad,2021,12,December,Thursday,4,Medium,False,False
5,T12465,U042,2021-12-16,Expense,food,4168,Unknown,not specified,2021,12,December,Thursday,4,Medium,False,False


#Creating Aggregated Features

In [ ]:
# Calculate user's total spending
user_totals = df_clean.groupby('user_id')['amount'].sum().reset_index()
user_totals.columns = ['user_id', 'total_user_spending']

# Merge back to original dataframe
df_clean = df_clean.merge(user_totals, on='user_id', how='left')

# Calculate user's average transaction
user_avg = df_clean.groupby('user_id')['amount'].mean().reset_index()
user_avg.columns = ['user_id', 'avg_user_transaction']

df_clean = df_clean.merge(user_avg, on='user_id', how='left')

# Now each row has the user's total and average spending!
print(df_clean[['user_id', 'amount', 'total_user_spending', 'avg_user_transaction']].head(10))

  user_id  amount  total_user_spending  avg_user_transaction
0    U018    3888               759875           8259.510870
1    U133     649               998527          12030.445783
2    U091   13239              1127019          11500.193878
3    U088    2287              1144448          12439.652174
4    U042    4168              1015065          11803.081395
5    U026     828              1137015          11370.150000
6    U061   62061              1057802          12020.477273
7    U053    5070              1311723          13249.727273
8    U004   59543              1334617          12132.881818
9    U112    4262               825539           9381.125000


In [ ]:
df_clean.head()

,transaction_id,user_id,date,transaction_type,category,amount,payment_mode,location,year,month,month_name,day_of_week,quarter,spending_level,is_high_value,is_weekend,total_user_spending,avg_user_transaction
0,T4999,U018,2023-04-25,Expense,education,3888,card,ahmedabad,2023,4,April,Tuesday,2,Medium,False,False,759875,8259.510870
1,T12828,U133,2021-12-16,Expense,rent,649,Unknown,hyderabad,2021,12,December,Thursday,4,Small,False,False,998527,12030.445783
2,T7403,U091,2021-12-16,Income,freelance,13239,Csh,bangalore,2021,12,December,Thursday,4,Large,True,False,1127019,11500.193878
3,T7495,U088,2021-12-16,Expense,entertainment,2287,CARD,hyderabad,2021,12,December,Thursday,4,Medium,False,False,1144448,12439.652174
4,T12465,U042,2021-12-16,Expense,food,4168,Unknown,not specified,2021,12,December,Thursday,4,Medium,False,False,1015065,11803.081395


#Multiple aggregations at once

In [ ]:
# Different metrics for each category
category_analysis = df_clean.groupby('category')['amount'].agg([
    ('total_spending', 'sum'),
    ('avg_spending', 'mean'),
    ('max_transaction', 'max'),
    ('min_transaction', 'min'),
    ('num_transactions', 'count'),
    ('median_spending', 'median')
]).reset_index()

category_analysis = category_analysis.sort_values('total_spending', ascending=False)
print(category_analysis)

         category  total_spending  avg_spending  max_transaction  \
9            rent        32870405  13764.826214            49978   
8          others        19351352  32306.096828            79870   
6      investment        19271991  46105.241627            79950   
4       freelance        18187518  44907.451852            79791   
0           bonus        17309995  46407.493298            79859   
10         salary        17094905  45708.302139            79999   
3            food        11022283   3691.320496            10000   
12         travel         8958610   5035.756043             9995   
2   entertainment         5962890   5070.484694             9999   
13      utilities         5513959   4963.059406             9990   
1       education         4705248   5210.684385             9998   
5          health         2715832   4919.985507             9979   
11        savings         1828835   5240.214900             9976   
14        utility         1487588   4958.626667 

#Saving Your Cleaned Data

**Exporting Clean Data**

After all that work cleaning your data, you need to save it!

In [ ]:
# Save to CSV
df_clean.to_csv('budgetwise_cleaned.csv', index=False)


In [ ]:
# Assuming your Google Drive is already mounted:

# 1. Define the full path for the new file
gdrive_path = '/content/drive/MyDrive/Colab Notebooks/WEEK2SESSION4/budgetwise_cleaned.csv'

# 2. Save the DataFrame to the new path
df_clean.to_csv(gdrive_path, index=False)

print(f"A new permanent file has been saved to: {gdrive_path}")

A new permanent file has been saved to: /content/drive/MyDrive/Colab Notebooks/WEEK2SESSION4/budgetwise_cleaned.csv


#SUMMARY
```
import pandas as pd
import numpy as np

# 1. LOAD DATA
df = pd.read_csv('budgetwise_finance_dataset.csv')
print(f"Original shape: {df.shape}")

# 2. HANDLE MISSING VALUES
df = df.drop(columns=['notes'])  # Too many missing
df['payment_mode'].fillna('Unknown', inplace=True)
df['location'].fillna('Not Specified', inplace=True)
df = df.dropna(subset=['date', 'category', 'amount'])

# 3. REMOVE DUPLICATES
df = df.drop_duplicates(subset=['transaction_id'])

# 4. CLEAN DATES
df['date'] = pd.to_datetime(df['date'], errors='coerce')
df = df.dropna(subset=['date'])

# 5. CLEAN AMOUNTS
df['amount'] = df['amount'].astype(str)
df['amount'] = df['amount'].str.replace('₹', '', regex=False)
df['amount'] = df['amount'].str.replace('$', '', regex=False)
df['amount'] = df['amount'].str.replace('Rs.', '', regex=False)
df['amount'] = df['amount'].str.replace(' INR', '', regex=False)
df['amount'] = df['amount'].str.replace(',', '', regex=False)
df['amount'] = df['amount'].str.strip()
df['amount'] = pd.to_numeric(df['amount'], errors='coerce')
df = df.dropna(subset=['amount'])
df = df[df['amount'] < 100000]  # Remove outliers

# 6. STANDARDIZE CATEGORIES
df['category'] = df['category'].str.lower().str.strip()
category_mapping = {
    'fod': 'food', 'foodd': 'food', 'foods': 'food',
    'rnt': 'rent', 'rentt': 'rent',
    'educaton': 'education', 'edu': 'education',
    'helth': 'health',
    'utlities': 'utilities', 'utilties': 'utilities',
    'entrtnmnt': 'entertainment', 'entertain': 'entertainment',
    'traval': 'travel', 'travl': 'travel'
}
df['category'] = df['category'].replace(category_mapping)

# 7. STANDARDIZE LOCATIONS
df['location'] = df['location'].str.lower().str.strip()
location_mapping = {
    'ban': 'bangalore', 'mum': 'mumbai', 'pun': 'pune',
    'del': 'delhi', 'che': 'chennai', 'hyd': 'hyderabad',
    'jai': 'jaipur', 'kol': 'kolkata', 'luc': 'lucknow',
    'ahm': 'ahmedabad'
}
df['location'] = df['location'].replace(location_mapping)

# 8. CREATE NEW FEATURES
df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month
df['month_name'] = df['date'].dt.month_name()
df['day_of_week'] = df['date'].dt.day_name()
df['quarter'] = df['date'].dt.quarter


df['is_weekend'] = df['day_of_week'].isin(['Saturday', 'Sunday'])

# 9. SAVE CLEANED DATA
df.to_csv('budgetwise_cleaned.csv', index=False)

print(f"Final shape: {df.shape}")
print(f"Data cleaning complete!")

# ASSIGNMENT.

```
Student Dataset:https://www.kaggle.com/datasets/samanfatima7/warehouse-and-retail-sales-montgomery-county


Handle missing values and duplicates.


Create at least two new derived columns (e.g., performance ratios or total campaign success).


Save the cleaned dataset as a new CSV and commit to GitHub.
